# Pipeline entregable FAT2019

Este notebook deja codificado el recorrido completo desde los audios crudos hasta el CSV final defendible.

La entrega final usa un blend simple de tres componentes reales:

```text
1/3 * separable_headsep
+ 1/3 * globalmel_sep_temporal
+ 1/3 * sep_temporal_f1024
```

Score Kaggle private verificado: `0.66649`.

Importante: este notebook esta preparado para correr, pero no fue ejecutado al crearlo. Los entrenamientos son largos; por defecto quedan desactivados con `RUN_HEAVY_STEPS = False`.

## 0. Configuracion

El flag `RUN_HEAVY_STEPS` controla si se reconstruyen caches y se reentrenan los modelos.

- `False`: modo revision/entrega. Usa los artefactos ya generados en `investigation/` para reconstruir y validar el CSV final.
- `True`: modo reproduccion desde cero. Genera caches log-mel y entrena las tres ramas finales desde los audios crudos.

La carpeta de salida de este notebook es `100. Entregable/outputs/`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

RUN_HEAVY_STEPS = False
RUN_OPTIONAL_KAGGLE_QUERY = False

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "train_curated.csv").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("No pude encontrar la raiz del repo con data/train_curated.csv")
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
INVESTIGATION = ROOT / "investigation"
SCRIPTS = INVESTIGATION / "scripts"
DELIVERABLE = ROOT / "100. Entregable"
OUTPUTS = DELIVERABLE / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

PYTHON = sys.executable
ENV = {"PYTHONPATH": str(INVESTIGATION)}
if (ROOT / "keys" / "kaggle.json").exists():
    ENV["KAGGLE_CONFIG_DIR"] = str(ROOT / "keys")

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def run_command(args: list[str | Path], *, heavy: bool = False) -> None:
    rendered = " ".join(str(arg) for arg in args)
    print(f"$ {rendered}")
    if heavy and not RUN_HEAVY_STEPS:
        print("SKIP: RUN_HEAVY_STEPS=False")
        return
    env = None
    if ENV:
        import os
        env = os.environ.copy()
        env.update(ENV)
    subprocess.run([str(arg) for arg in args], check=True, cwd=ROOT, env=env)

print("ROOT", ROOT)
print("RUN_HEAVY_STEPS", RUN_HEAVY_STEPS)

## 1. Cargar y validar los datos crudos

La competencia entrega audios `.wav`, `train_curated.csv` con etiquetas multilabel y `sample_submission.csv` con el formato esperado.

Esta celda verifica que las filas referenciadas por los CSV tengan archivos de audio disponibles y que el formato base tenga 80 clases.

In [ ]:
train_curated = pd.read_csv(DATA_DIR / "train_curated.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
label_columns = list(sample_submission.columns[1:])

train_missing = [fname for fname in train_curated["fname"].astype(str) if not (DATA_DIR / fname).exists()]
test_missing = [fname for fname in sample_submission["fname"].astype(str) if not (DATA_DIR / fname).exists()]

summary = {
    "train_rows": len(train_curated),
    "test_rows": len(sample_submission),
    "num_classes": len(label_columns),
    "wav_files_in_data": len(list(DATA_DIR.glob("*.wav"))),
    "missing_train_audio": len(train_missing),
    "missing_test_audio": len(test_missing),
}
summary

## 2. Preprocesamiento: audios a espectrogramas log-mel

Los tres modelos finales trabajan sobre imagenes log-mel. El script `build_logmel_image_cache.py` hace:

```text
wav -> waveform mono -> resample 44.1 kHz -> MelSpectrogram -> dB -> crop/pad temporal -> normalizacion -> .npz
```

Se necesitan tres caches:

1. `m128_f512` con normalizacion por clip para `separable_headsep`.
2. `m128_f512_globalmel` con media/desvio global por banda mel para `globalmel_sep_temporal`.
3. `m128_f1024` con mas contexto temporal para `sep_temporal_f1024`.

In [ ]:
# Cache base: 128 bandas mel, 512 frames, normalizacion por clip.
run_command([
    PYTHON, SCRIPTS / "build_logmel_image_cache.py",
    "--data-dir", DATA_DIR,
    "--n-mels", "128",
    "--frames", "512",
    "--normalization", "clip",
], heavy=True)

# Cache globalmel: mismas dimensiones, pero normalizacion global por banda mel.
run_command([
    PYTHON, SCRIPTS / "build_logmel_image_cache.py",
    "--data-dir", DATA_DIR,
    "--n-mels", "128",
    "--frames", "512",
    "--normalization", "global-mel",
    "--cache-tag", "globalmel",
], heavy=True)

# Cache temporal larga: 128 bandas mel, 1024 frames.
run_command([
    PYTHON, SCRIPTS / "build_logmel_image_cache.py",
    "--data-dir", DATA_DIR,
    "--n-mels", "128",
    "--frames", "1024",
    "--normalization", "clip",
], heavy=True)

## 3. Entrenar componente 1: `separable_headsep`

`separable_headsep` es la rama CNN fuerte. Usa el espectrograma como imagen tiempo-frecuencia y aplica bloques separables residuales.

Configuracion principal:

- entrada log-mel `128 x 512`;
- arquitectura `separable_residual`;
- activacion `relu`;
- inicializacion He;
- cabeza densa de 256 unidades;
- dropout de cabeza `0.3`;
- entrenamiento full-train de 56 epochs.

Rol en el blend: capturar patrones locales tiempo-frecuencia con una CNN fuerte.

In [ ]:
separable_headsep_dir = OUTPUTS / "separable_headsep"
run_command([
    PYTHON, SCRIPTS / "train_logmel_cnn.py",
    "--data-dir", DATA_DIR,
    "--models-dir", separable_headsep_dir / "models",
    "--submissions-dir", separable_headsep_dir / "submissions",
    "--experiments-dir", separable_headsep_dir / "experiments",
    "--n-mels", "128",
    "--frames", "512",
    "--architecture", "separable_residual",
    "--activation", "relu",
    "--initializer", "he_normal",
    "--head-hidden", "256",
    "--head-dropout", "0.3",
    "--optimizer", "adam",
    "--scheduler", "multistep",
    "--lr-milestones", "27,36,43,49,52",
    "--epochs", "56",
    "--batch-size", "24",
    "--seed", "42",
    "--full-train",
], heavy=True)

## 4. Entrenar componente 2: `globalmel_sep_temporal`

Esta rama usa la misma familia separable, pero agrega una cabeza temporal BiGRU. La diferencia clave es el preprocesamiento: en vez de normalizar cada clip de forma independiente, normaliza cada banda mel con estadisticas globales calculadas sobre `train_curated`.

Rol en el blend: aportar una vista temporal con una normalizacion mas estable entre audios.

In [ ]:
globalmel_dir = OUTPUTS / "globalmel_sep_temporal"
run_command([
    PYTHON, SCRIPTS / "train_logmel_cnn.py",
    "--data-dir", DATA_DIR,
    "--models-dir", globalmel_dir / "models",
    "--submissions-dir", globalmel_dir / "submissions",
    "--experiments-dir", globalmel_dir / "experiments",
    "--n-mels", "128",
    "--frames", "512",
    "--cache-tag", "globalmel",
    "--architecture", "separable_temporal_bigru",
    "--activation", "silu",
    "--initializer", "he_normal",
    "--head-dropout", "0.3",
    "--optimizer", "adamw",
    "--scheduler", "multistep",
    "--lr-milestones", "25,39",
    "--epochs", "40",
    "--batch-size", "24",
    "--seed", "42",
    "--full-train",
], heavy=True)

## 5. Entrenar componente 3: `sep_temporal_f1024`

Esta rama mantiene la arquitectura temporal CNN + BiGRU, pero aumenta la ventana de entrada a `1024` frames.

Rol en el blend: capturar eventos o patrones sonoros que necesitan mas contexto temporal que una ventana de 512 frames.

In [ ]:
f1024_dir = OUTPUTS / "sep_temporal_f1024"
run_command([
    PYTHON, SCRIPTS / "train_logmel_cnn.py",
    "--data-dir", DATA_DIR,
    "--models-dir", f1024_dir / "models",
    "--submissions-dir", f1024_dir / "submissions",
    "--experiments-dir", f1024_dir / "experiments",
    "--n-mels", "128",
    "--frames", "1024",
    "--architecture", "separable_temporal_bigru",
    "--activation", "silu",
    "--initializer", "he_normal",
    "--head-dropout", "0.3",
    "--optimizer", "adamw",
    "--scheduler", "multistep",
    "--lr-milestones", "19,25",
    "--epochs", "40",
    "--batch-size", "12",
    "--seed", "42",
    "--full-train",
], heavy=True)

## 6. Elegir artefactos para el blend

Si `RUN_HEAVY_STEPS=True`, se usan las predicciones recien entrenadas en `100. Entregable/outputs/`.

Si `RUN_HEAVY_STEPS=False`, se usan los artefactos ya generados y validados en `investigation/`. Esto permite reconstruir el CSV final sin reentrenar.

In [ ]:
if RUN_HEAVY_STEPS:
    component_submissions = {
        "separable_headsep": separable_headsep_dir / "submissions" / "small_logmel_cnn.csv",
        "globalmel_sep_temporal": globalmel_dir / "submissions" / "small_logmel_cnn.csv",
        "sep_temporal_f1024": f1024_dir / "submissions" / "small_logmel_cnn.csv",
    }
else:
    component_submissions = {
        "separable_headsep": ROOT / "investigation/submissions/logmel_cnn_catsdogs_sepres_head256_full_e56_seed42/small_logmel_cnn.csv",
        "globalmel_sep_temporal": ROOT / "investigation/submissions/theory_globalmel_sep_temporal_full_e40_seed42/small_logmel_cnn.csv",
        "sep_temporal_f1024": ROOT / "investigation/submissions/theory_sep_temporal_f1024_full_e40_seed42/small_logmel_cnn.csv",
    }

for name, path in component_submissions.items():
    print(name, path, path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

## 7. Blend final

El blend es un soft vote con pesos iguales. Es intencionalmente simple: cada componente cuenta un tercio y todos son modelos reales, no ensembles anidados.

In [ ]:
final_submission_path = DELIVERABLE / "submission.csv"

blend_args = [
    PYTHON, SCRIPTS / "blend_submissions.py",
    "--input", component_submissions["separable_headsep"], "--weight", "1",
    "--input", component_submissions["globalmel_sep_temporal"], "--weight", "1",
    "--input", component_submissions["sep_temporal_f1024"], "--weight", "1",
    "--output", final_submission_path,
]
run_command(blend_args)

print("final", final_submission_path)
print("sha256", sha256(final_submission_path))

## 8. Validacion de formato y equivalencia

La submission final debe respetar exactamente `sample_submission.csv`: mismas filas, mismas columnas y probabilidades en `[0, 1]`.

Ademas, cuando `RUN_HEAVY_STEPS=False`, el CSV reconstruido debe coincidir con el artefacto final guardado en `04_final/submission.csv`.

In [ ]:
final_df = pd.read_csv(final_submission_path)
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
label_columns = list(sample_df.columns[1:])

assert list(final_df.columns) == list(sample_df.columns)
assert final_df["fname"].equals(sample_df["fname"])
assert final_df[label_columns].ge(0.0).all().all()
assert final_df[label_columns].le(1.0).all().all()

reference_final = ROOT / "04_final" / "submission.csv"
validation = {
    "rows": len(final_df),
    "columns": len(final_df.columns),
    "min_probability": float(final_df[label_columns].min().min()),
    "max_probability": float(final_df[label_columns].max().max()),
    "sha256": sha256(final_submission_path),
    "matches_04_final_submission": reference_final.exists() and sha256(final_submission_path) == sha256(reference_final),
}
validation

## 9. Evaluacion

La evaluacion final real se hizo en Kaggle, porque el test no tiene etiquetas publicas locales.

Resultado registrado para este CSV:

```text
description: simple current-free headsep globalmel f1024 equal
private LB: 0.66649
public LB: 0.00000  # la API devuelve 0.00000 para estas submissions post-competencia
```

La celda siguiente permite consultar la tabla de Kaggle si existen credenciales en `keys/`. Por defecto esta desactivada.

In [ ]:
if RUN_OPTIONAL_KAGGLE_QUERY:
    run_command([
        "kaggle", "competitions", "submissions",
        "-c", "freesound-audio-tagging-2019",
    ])
else:
    print("Kaggle query desactivada. Score registrado: private LB 0.66649.")

## 10. Resumen final

El pipeline completo produce `100. Entregable/submission.csv`.

La decision final prioriza presentabilidad y trazabilidad:

- 3 componentes reales;
- sin `current` anidado;
- pesos iguales;
- todos los componentes operan sobre espectrogramas log-mel;
- Kaggle private LB `0.66649`.

El mejor historico expandido (`0.67025`) queda como referencia, pero no como entrega principal porque contiene un ensemble anidado de 10 componentes reales.